In [1]:
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len = 5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2)
            * (-math.log(10000.0) /embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
    
    def forward(self,x, mask):
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(-2,-1)
        scores /= math.sqrt(self.head_dim)
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim= -1)
        output = weights @ v
        output = output.transpose(1,2).contiguous()
        output = output.view(batch_size, seq_len, self.embed_dim)
        output = self.out_proj(output)
        
        return output

In [4]:
class CrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, decoder_x, encoder_output, mask):
        batch_size, seq_len, _ = decoder_x.shape
        _, encoder_len,_ = encoder_output.shape
        q = self.q_proj(decoder_x)
        k = self.k_proj(encoder_output)
        v = self.v_proj(encoder_output)
        
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, encoder_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, encoder_len, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(-2,-1)
        scores /= math.sqrt(self.head_dim)
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim= -1)
        output = weights @ v
        output = output.transpose(1,2).contiguous()
        output = output.view(batch_size, seq_len, self.embed_dim)
        output = self.out_proj(output)
        
        return output

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x, mask):
        x = x + self.attention(x, mask)
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x

In [6]:
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
    
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [7]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.mask_attention = MultiHeadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)

        self.cross_attention = CrossAttention(embed_dim, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm3 = nn.LayerNorm(embed_dim)
    
    def forward(self, x,encoder_output, tgt_mask, src_mask):
        x = x + self.mask_attention(x, tgt_mask)
        x = self.norm1(x)
        x = self.cross_attention(x, encoder_output, src_mask)
        x = self.norm2(x)
        x = x + self.ffn(x)
        x = self.norm3(x)
        return x
        

In [8]:
class TransformerDecoder(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
    
    def forward(self, x, encoder_output, tgt_mask, src_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, tgt_mask, src_mask)
        return x

In [9]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size,tgt_vocab_size, embed_dim, num_heads, num_layers):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        
        self.pos_encoding = PositionalEncoding(embed_dim)
        self.encoder = TransformerEncoder(
            embed_dim,
            num_heads,
            num_layers
        )
        self.decoder = TransformerDecoder(
            embed_dim,
            num_heads,
            num_layers
        )
        
        self.fc = nn.Linear(embed_dim, tgt_vocab_size)
    
    def forward(self, src, tgt, src_mask, tgt_mask):
        src = self.src_embedding(src)
        src = self.pos_encoding(src)
        encoder_output = self.encoder(src, src_mask)

        tgt = self.tgt_embedding(tgt)
        tgt = self.pos_encoding(tgt)
        decoder_output = self.decoder(tgt, encoder_output,tgt_mask, src_mask)
        output = self.fc(decoder_output)
        return output

In [10]:
class TranslationDataset(Dataset):

    def __init__(self, src_texts, tgt_texts, tokenizer):

        self.src = src_texts
        self.tgt = tgt_texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):

        src = self.tokenizer(
            self.src[idx],
            padding="max_length",
            truncation=True,
            max_length= 64,
            return_tensors="pt"
        )

        tgt = self.tokenizer(
            self.tgt[idx],
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        src_ids = src["input_ids"].squeeze(0)

        tgt_ids = tgt["input_ids"].squeeze(0)

        tgt_input = tgt_ids[:-1]

        tgt_output = tgt_ids[1:]

        return (
            src_ids,
            tgt_input,
            tgt_output
        )

In [11]:
from datasets import load_dataset

dataset = load_dataset(
    "opus_books",
    "en-fr"
)

src_texts = [
    x["translation"]["en"]
    for x in dataset["train"]
]

tgt_texts = [
    x["translation"]["fr"]
    for x in dataset["train"]
]

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

train_dataset = TranslationDataset(
    src_texts,
    tgt_texts,
    tokenizer
)

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size= 16,
    shuffle=True
)


README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
def create_src_mask(src):

    return (src != 0).long()

def create_tgt_mask(tgt):

    seq_len = tgt.size(1)

    pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)

    causal_mask = torch.tril(
        torch.ones(seq_len, seq_len, device=tgt.device)
    ).bool()

    return pad_mask & causal_mask

In [13]:

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Transformer(
    src_vocab_size=tokenizer.vocab_size,
    tgt_vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    num_heads=8,
    num_layers=6
).to(device)

loss_fn = nn.CrossEntropyLoss(
    ignore_index=tokenizer.pad_token_id
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [14]:
for epoch in range(10):

    model.train()

    total_loss = 0

    for src, tgt_input, tgt_output in train_loader:

        src = src.to(device)
        tgt_input = tgt_input.to(device)
        tgt_output = tgt_output.to(device)

        src_mask = create_src_mask(src)
        tgt_mask = create_tgt_mask(tgt_input)

        pred = model(
            src,
            tgt_input,
            src_mask,
            tgt_mask
        )

        loss = loss_fn(
            pred.reshape(-1, pred.size(-1)),
            tgt_output.reshape(-1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{total_loss / len(train_loader):.4f}"
    )

Epoch 1: 6.5835
Epoch 2: 6.2188
Epoch 3: 6.0818
Epoch 4: 5.9801
Epoch 5: 5.8985
Epoch 6: 5.8296
Epoch 7: 5.7705
Epoch 8: 5.7186
Epoch 9: 5.6724
Epoch 10: 5.6304


In [15]:
def translate(
    sentence,
    model,
    src_tokenizer,
    tgt_tokenizer,
    device,
    max_len=50
):

    model.eval()

    src = src_tokenizer(
        sentence,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=64
    )

    src_ids = src["input_ids"].to(device)

    src_mask = create_src_mask(src_ids)

    tgt_ids = torch.tensor(
        [[tgt_tokenizer.cls_token_id]],
        device=device
    )

    with torch.no_grad():

        for _ in range(max_len):

            tgt_mask = create_tgt_mask(tgt_ids)

            output = model(
                src_ids,
                tgt_ids,
                src_mask,
                tgt_mask
            )

            next_token = output[:, -1].argmax(dim=-1)

            tgt_ids = torch.cat(
                [tgt_ids, next_token.unsqueeze(1)],
                dim=1
            )

            if next_token.item() == tgt_tokenizer.sep_token_id:
                break

    return tgt_tokenizer.decode(
        tgt_ids[0],
        skip_special_tokens=True
    )

In [16]:
sentences = [
    "Hello, how are you today?",
    "This is a simple test.",
    "I love reading books in the library."
]

print("-------Result-------")
for text in sentences:
    result = translate(text, model, tokenizer, tokenizer, device)
    print(f"EN: {text}")
    print(f"FR: {result}")
    print("-" * 30)

-------Result-------
EN: Hello, how are you today?
FR: 
------------------------------
EN: This is a simple test.
FR: simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple simple
------------------------------
EN: I love reading books in the library.
FR: 
------------------------------
